In [ ]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from langchain.tools import tool, ToolRuntime
import time
from dotenv import load_dotenv


In [ ]:
load_dotenv()

# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = os.getenv('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = os.getenv('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = os.getenv('LANGSMITH_PROJECT')
os.environ['DEEPSEEK_API_KEY'] = os.getenv('DEEPSEEK_API_KEY')

- Simplest way to create a tool is with the @tool decorator.

- The function's docstring becomes the tool's description that helps the model understand when to use it.

- Type hints (def function(expression:str) -> str) **must** be provided as they define the tool's input schema.

In [ ]:
# Creating a basic tool
# Modify the answer by removing the caps lock text.
# RETURN THE ANSWER AS A SINGLE VALUE WITH NO ADDITIONAL TEXT
@tool("multiplication_calculator") # -> Defines tool name
def multiplication_calculator(expression:str, runtime:ToolRuntime)->str:
  """Evaluate a mathematical expression and return the result.

  Args:
    expresion: A mathematical expression like:
       - "y by z", or
       - "y*z", or
       - "y multiplied by z", or
       - "y times z", or
       - "y x z"

    In which case, y and z are both numbers, whether floats or integers.

    Before entering the expression into the function, make sure that this rewritten, if necessary, into the format y*z.



  Returns:
    Dictionary with 'result' or 'error'
  """

  # Add logs execution identity information
  info = runtime.execution_info
  print(f"Thread: {info.thread_id}, Run: {info.run_id}")
  print(f"Attempt: {info.node_attempt}")

  try:
    # Evaluate the expression
    result = eval(expression)
    return {
        "success'": True,
        "result": result,
        "expression": expression
    }
  except Exception as e:
    return {
        "success": False,
        "error": str(e),
        "expression":expression
    }

In [ ]:
ai_context = """You are a helpful assistant"""

human_message = """What is 12836458 times 192865?"""


In [ ]:
agent = create_agent(model = "deepseek:deepseek-v4-flash", tools=[multiplication_calculator])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)

In [ ]:
print(result)

In [ ]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
output_list

In [ ]:
# If you want a neater output
print(output_list[1])

In [ ]:
# Let's try something more interesting
# Let's try something more interesting
from bs4 import BeautifulSoup
import re
import urllib3

@tool("weather_page")
def get_weather(expression: str) -> str:
  """Gets the current weather for Berlin by scraping weather data from HTML response.

  Args:
    expression: The user's weather query (e.g., "What's the weather in Berlin?")

  Returns:
    A string containing the parsed weather information or an error message
  """

  try:
      # Create a PoolManager instance for sending requests
      http = urllib3.PoolManager()
      url = 'https://www.theweathernetwork.com/en/city/de/berlin/berlin/current'

      # Send GET request
      resp = http.request("GET", url)

      # Parse HTML with BeautifulSoup
      soup = BeautifulSoup(resp.data, 'html.parser')

      return soup

  except Exception as e:
      return f"Error fetching weather: {str(e)}"

In [ ]:

ai_context = """You are a helpful assistant"""

human_message = """What's the weather in Berlin like?"""

agent = create_agent(model = "deepseek:deepseek-v4-flash", tools=[get_weather])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)

In [ ]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
for i in range(len(output_list)):
    print(output_list[i])

In [ ]:

ai_context = """You are a helpful assistant"""

human_message = """What's the weather in Berlin like? Get it in celcius and return it to me in Fahrenheit by multiplying by 1.8 and adding the outcome to 32"""

agent = create_agent(model = "deepseek:deepseek-v4-flash", tools=[get_weather, multiplication_calculator])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)

In [ ]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
for i in range(len(output_list)):
    print(output_list[i])